# Graph Neural Network (GNN)

## Part I) GIN implementation :

Implement a graph isomorphism network model (GIN), then test your implementation on the PROTEINS dataset.

The task is to predict if a protein is an enzyme or not. It's a graph level binary classification task.

To install torch_geometric : pip install torch_geometric

More info on : https://pytorch-geometric.readthedocs.io/en/latest/install/installation.html

In [2]:
#main import function and library
import torch_geometric as tg
import numpy as np
import torch
from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch.nn import Linear, Parameter
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv,global_mean_pool
from torch.nn import Sequential, Linear, ReLU



In [3]:
dataset = tg.datasets.TUDataset(root='/tmp/PROTEINS', name='PROTEINS', use_node_attr=True)
dataset = dataset.shuffle()
train_dataset = dataset[:800]
val_dataset = dataset[800:956]
test_dataset = dataset[956:]


print('Number of training graphs: ',len(train_dataset))
print('Number of validation graphs: ',len(test_dataset))
print('Number of test graphs: ',len(test_dataset))

Number of training graphs:  800
Number of validation graphs:  157
Number of test graphs:  157


In [17]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [8]:
#GIN model with pytorch geometric

class GIN(torch.nn.Module):
    def __init__(self):
        super().__init__()

        nn1 = Sequential(
            Linear(dataset.num_features, 32),
            ReLU(),
            Linear(32, 32)
        )
        self.conv1 = GINConv(nn1)

        nn2 = Sequential(
            Linear(32, 32),
            ReLU(),
            Linear(32, 32)
        )
        self.conv2 = GINConv(nn2)

        # Final binary output → 1 logit
        self.lin = Linear(32, 1)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        # GRAPH embedding (one per graph)
        x = global_mean_pool(x, batch)

        # one logit per graph
        logits = self.lin(x)

        return logits.view(-1)  # shape: (batch_size,)





In [12]:
model=GIN()
optimizer=torch.optim.RMSprop(model.parameters(), lr=0.001)
criterion = torch.nn.BCEWithLogitsLoss()


In [13]:

def train(model,loader):
    model.train()
    total_loss = 0

    for data in loader:        # iterate on BATCHES
        optimizer.zero_grad()
        out = model(data)      # now model accepts Data object
        y = data.y.float() 
        loss = criterion(out,y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


for epoch in range(1, 201):
    loss = train(model, train_loader)
    print(f"Epoch {epoch}: loss={loss:.4f}")

Epoch 1: loss=0.7042
Epoch 2: loss=0.6775
Epoch 3: loss=0.6778
Epoch 4: loss=0.6767
Epoch 5: loss=0.6724
Epoch 6: loss=0.6676
Epoch 7: loss=0.6673
Epoch 8: loss=0.6635
Epoch 9: loss=0.6552
Epoch 10: loss=0.6500
Epoch 11: loss=0.6509
Epoch 12: loss=0.6326
Epoch 13: loss=0.6372
Epoch 14: loss=0.6352
Epoch 15: loss=0.6305
Epoch 16: loss=0.6334
Epoch 17: loss=0.6278
Epoch 18: loss=0.6233
Epoch 19: loss=0.6217
Epoch 20: loss=0.6193
Epoch 21: loss=0.6182
Epoch 22: loss=0.6137
Epoch 23: loss=0.6147
Epoch 24: loss=0.6155
Epoch 25: loss=0.6068
Epoch 26: loss=0.6180
Epoch 27: loss=0.6098
Epoch 28: loss=0.6044
Epoch 29: loss=0.6028
Epoch 30: loss=0.6038
Epoch 31: loss=0.6038
Epoch 32: loss=0.6017
Epoch 33: loss=0.5997
Epoch 34: loss=0.6030
Epoch 35: loss=0.5986
Epoch 36: loss=0.5997
Epoch 37: loss=0.6050
Epoch 38: loss=0.5969
Epoch 39: loss=0.6046
Epoch 40: loss=0.6007
Epoch 41: loss=0.5960
Epoch 42: loss=0.5913
Epoch 43: loss=0.5997
Epoch 44: loss=0.5981
Epoch 45: loss=0.5974
Epoch 46: loss=0.59

In [14]:
def test(model,loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for data in loader:
            logits = model(data)                    
            probs = torch.sigmoid(logits)          
            preds = (probs > 0.5).long()            

            correct += (preds == data.y).sum().item()
            total += data.y.size(0)

    return correct / total

In [19]:
test(model, test_loader)

0.7643312101910829

In [ ]:
#Optionnal
#Specified layer

class MyLayer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = nn.Linear(in_channels, out_channels)
        self.reset_parameters()

    def reset_parameters(self):
        self.lin.reset_parameters()

    def forward(self, x, edge_index):
      
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        return self.propagate(edge_index, x=x)

    def message(self, x_j):
        return self.lin(x_j)

    def update(self, aggr_out):
        return aggr_out




In [ ]:
class GINMyLayer(torch.nn.Module):
    
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = MyLayer(in_channels, hidden_channels)
        self.conv2 = MyLayer(hidden_channels, out_channels)
        self.lin = Linear(out_channels, 1)
        
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))

        
        x = global_mean_pool(x, batch)

   
        logits = self.lin(x)

        return logits.view(-1) 


In [59]:
model2=GINMyLayer(dataset.num_features,32,32)

In [61]:
# Training function

for epoch in range(1, 201):
    loss = train(model2,train_loader)
    print(f"Epoch {epoch}: loss={loss:.4f}")

Epoch 1: loss=1.0202
Epoch 2: loss=1.0202
Epoch 3: loss=1.0202
Epoch 4: loss=1.0202
Epoch 5: loss=1.0202
Epoch 6: loss=1.0202
Epoch 7: loss=1.0202
Epoch 8: loss=1.0202
Epoch 9: loss=1.0202
Epoch 10: loss=1.0202
Epoch 11: loss=1.0202
Epoch 12: loss=1.0202
Epoch 13: loss=1.0202
Epoch 14: loss=1.0202
Epoch 15: loss=1.0202
Epoch 16: loss=1.0202
Epoch 17: loss=1.0202
Epoch 18: loss=1.0202
Epoch 19: loss=1.0202
Epoch 20: loss=1.0202
Epoch 21: loss=1.0202
Epoch 22: loss=1.0202
Epoch 23: loss=1.0202
Epoch 24: loss=1.0202
Epoch 25: loss=1.0202
Epoch 26: loss=1.0202
Epoch 27: loss=1.0202
Epoch 28: loss=1.0202
Epoch 29: loss=1.0202
Epoch 30: loss=1.0202
Epoch 31: loss=1.0202
Epoch 32: loss=1.0202
Epoch 33: loss=1.0202
Epoch 34: loss=1.0202
Epoch 35: loss=1.0202
Epoch 36: loss=1.0202
Epoch 37: loss=1.0202
Epoch 38: loss=1.0202
Epoch 39: loss=1.0202
Epoch 40: loss=1.0202
Epoch 41: loss=1.0202
Epoch 42: loss=1.0202
Epoch 43: loss=1.0202
Epoch 44: loss=1.0202
Epoch 45: loss=1.0202
Epoch 46: loss=1.02

In [65]:
test(model2,test_loader)


0.6242038216560509

Explain what does the Aggregation and Update functions of GIN? Why it's called a message-passing architecture?

## Part II) Graph learning challenge

Now you have to implement and train a GNN on the ZFR database, you can certainly test your GIN from part I) but also any GNN you want (GCN, GAT...).

The goal is to predict three properties per graph, it's a graph regression task.

First, predict each property separately, then all three properties together.

This part of the practical work will be assessed on the basis of your experimental protocol and the results obtained.

Once your GNN is ready, download the test database and make predictions using your GNN.

You should submit your GNN predictions in a .csv file to Universitice. Your predictions should be in the same order as the test dataset. Your predictions should respect the following csv format:

- separator ";"
- column names : "prop1","prop2", "prop3", "allprop".
- for "allprop" your predictions should be a list of three values per graphs.
- csv file name :  yourName_GNNmodelName.csv

Any csv files that do not respect the above format will not be processed.

An accessible sheet will be filled with your name and your results (MAE and model name only) and a baseline (GCN model). You can submit any attempts of predictions, just modify your file on Universitice. The results will be processed every week before the practical session.

https://docs.google.com/spreadsheets/d/1fTZyV3DSa39vVlEYxN747YIGSR7ONoYEQc4Z3CHW3uM/edit?gid=0#gid=0

For the final rending of this work, you must submit to Universitice: this notebook, the code that gave you the best MAE entry (in .py or .ipynb file), but also a doc with all your reasoning, methodology and different experiments you had done and why.

For example: the list of hyperparameters and a short description of the model used to obtain your best result, the hyperparameterisation protocol, the different sizes of architectures, the number of trials you performed, the learning protocol, the tested loss, the tested layers, the learning rate, the scheduler, the drop-out, etc...

The goal of this challenge is not just to code a GNN, but to understand how to train a deep neural network well on graph data.

The final grade for this work will focus mainly on your results, your originality and the protocol you used. Good luck!

In [2]:
# After dowloading the databases import ZFR to load datasets 
from ZFRdataset import ZFR
dataset_train = ZFR('dataset/ZFR',dataset = 'train')
dataset_test = ZFR('dataset/ZFR',dataset = 'test')

In [ ]:
#Important for reproducibility of your results
def set_seed(seed = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

In [3]:
print(dataset_test.data)

Data(x=[188602, 13], edge_index=[2, 382970], edge_attr=[382970, 4], pos=[188602, 3], z=[188602])


In [4]:
print(dataset_train.data)

Data(x=[860270, 13], edge_index=[2, 1789766], edge_attr=[1789766, 4], y=[48479, 3], pos=[860270, 3], z=[860270])
